# Aral Sea -- Water Extent Time Series (2005-2024)

The Aral Sea in Central Asia is one of the most dramatic examples of
human-induced environmental change. Fed by the Amu Darya and Syr Darya
rivers, it began shrinking in the 1960s after Soviet irrigation projects
diverted its inflows. By the 2010s the South Aral had almost completely
disappeared.

This notebook builds a multi-decadal time series using:
- **Landsat Collection 2 Level-2** (2005-2015) -- 30 m, ~16-day revisit
- **Sentinel-2 Level-2A** (2016 onwards) -- 10/20 m, ~5-day revisit

Both datasets are accessed live from **Microsoft Planetary Computer** via STAC --
no local download needed.

**Workflow:** search -> load -> spectral indices -> binary water mask -> timelapse GIF


## 1. Imports and configuration


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from io import BytesIO
import os

import pystac_client
import planetary_computer
import odc.stac
import imageio.v3 as iio

# ---------------------------------------------------------------------------
# Region of interest: Aral Sea basin (WGS84 lon_min, lat_min, lon_max, lat_max)
BBOX = [57.5, 43.5, 62.5, 46.8]

# Summer window -- minimal cloud cover over Central Asia
SEASON_START = "07-01"
SEASON_END   = "09-30"

# Maximum cloud cover per scene (%)
CLOUD_MAX = 15

# Spatial resolution for loading (metres).
# Increase to speed up loading, decrease for sharper maps.
RESOLUTION = 300

print(f"Region     : lon {BBOX[0]}-{BBOX[2]}, lat {BBOX[1]}-{BBOX[3]}")
print(f"Season     : {SEASON_START} to {SEASON_END}")
print(f"Cloud max  : {CLOUD_MAX} %")
print(f"Resolution : {RESOLUTION} m")


## 2. Connect to Planetary Computer

Planetary Computer is a free public STAC catalogue by Microsoft. It hosts
Landsat and Sentinel-2 global archives. Asset URLs must be *signed* before
downloading -- `planetary_computer.sign_inplace` handles this transparently.


In [ ]:
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)
print("Connected:", catalog.title)


## 3. Landsat search -- 2005 to 2015

**Collection:** `landsat-c2-l2` (Collection 2, Level-2 Surface Reflectance).  
Covers Landsat 5 TM, Landsat 7 ETM+ and Landsat 8 OLI.  
We select the least-cloudy summer scene per year and use band common names
(`green`, `nir08`, `swir16`) so the query works across all three sensors.


In [ ]:
landsat_items = {}

for year in range(2005, 2016):
    search = catalog.search(
        collections=["landsat-c2-l2"],
        bbox=BBOX,
        datetime=f"{year}-{SEASON_START}/{year}-{SEASON_END}",
        query={"eo:cloud_cover": {"lt": CLOUD_MAX}},
    )
    items = list(search.items())
    if not items:
        # Relax threshold if nothing found
        search = catalog.search(
            collections=["landsat-c2-l2"],
            bbox=BBOX,
            datetime=f"{year}-{SEASON_START}/{year}-{SEASON_END}",
            query={"eo:cloud_cover": {"lt": 40}},
        )
        items = list(search.items())

    if items:
        best = min(items, key=lambda x: x.properties.get("eo:cloud_cover", 100))
        landsat_items[year] = best
        platform = best.properties.get("platform", "?")
        cloud    = best.properties.get("eo:cloud_cover", float("nan"))
        print(f"{year}: {best.id[:45]}  [{platform}]  cloud={cloud:.1f}%")
    else:
        print(f"{year}: no scene found")

print(f"\nTotal Landsat scenes: {len(landsat_items)}")


### Load Landsat bands

Landsat C2 L2 Surface Reflectance is stored as uint16 with:
- scale factor: **0.0000275**
- additive offset: **-0.2**

=> reflectance = DN x 0.0000275 - 0.2

`odc.stac.load()` reprojects all scenes to the same UTM grid automatically,
handling the different native projections of Landsat path/rows.


In [ ]:
def apply_landsat_scale(ds):
    """Convert Landsat C2 L2 DN to surface reflectance."""
    return (ds * 0.0000275 - 0.2).clip(0, 1)


landsat_data = {}

for year, item in sorted(landsat_items.items()):
    print(f"Loading {year} ...", end=" ", flush=True)
    ds = odc.stac.load(
        [item],
        bands=["green", "nir08", "swir16"],
        bbox=BBOX,
        resolution=RESOLUTION,
        groupby="solar_day",
    )
    landsat_data[year] = apply_landsat_scale(ds.isel(time=0))
    print(f"{dict(ds.isel(time=0).dims)}")

print("Done.")


## 4. Sentinel-2 search -- 2016 to present

**Collection:** `sentinel-2-l2a` (Level-2A Surface Reflectance).  
Sentinel-2A launched April 2015; systematic global coverage starts 2016.  
Bands used: `green` (B03, 10 m), `nir08` (B08, 10 m), `swir16` (B11, 20 m).

One best-quality summer scene per year, same strategy as Landsat.


In [ ]:
s2_items = {}

for year in range(2016, 2025):
    search = catalog.search(
        collections=["sentinel-2-l2a"],
        bbox=BBOX,
        datetime=f"{year}-{SEASON_START}/{year}-{SEASON_END}",
        query={"eo:cloud_cover": {"lt": CLOUD_MAX}},
    )
    items = list(search.items())
    if not items:
        search = catalog.search(
            collections=["sentinel-2-l2a"],
            bbox=BBOX,
            datetime=f"{year}-{SEASON_START}/{year}-{SEASON_END}",
            query={"eo:cloud_cover": {"lt": 40}},
        )
        items = list(search.items())

    if items:
        best = min(items, key=lambda x: x.properties.get("eo:cloud_cover", 100))
        s2_items[year] = best
        cloud = best.properties.get("eo:cloud_cover", float("nan"))
        print(f"{year}: {best.id[:45]}  cloud={cloud:.1f}%")
    else:
        print(f"{year}: no scene found")

print(f"\nTotal Sentinel-2 scenes: {len(s2_items)}")


### Load Sentinel-2 bands

Sentinel-2 L2A reflectance values are scaled by **0.0001** (divide by 10 000).
`odc.stac` resamples B11 (native 20 m) to the target resolution automatically.


In [ ]:
s2_data = {}

for year, item in sorted(s2_items.items()):
    print(f"Loading {year} ...", end=" ", flush=True)
    ds = odc.stac.load(
        [item],
        bands=["green", "nir08", "swir16"],
        bbox=BBOX,
        resolution=RESOLUTION,
        groupby="solar_day",
    )
    s2_data[year] = (ds.isel(time=0) * 0.0001).clip(0, 1)
    print(f"{dict(ds.isel(time=0).dims)}")

print("Done.")


## 5. Spectral water indices

| Index | Formula | Notes |
|---|---|---|
| **NDWI** (McFeeters 1996) | (Green - NIR) / (Green + NIR) | General open-water detection |
| **MNDWI** (Xu 2006) | (Green - SWIR) / (Green + SWIR) | Better suppression of built-up / soil |
| **AWEInsh** (Feyisa 2014) | 4*(Green-SWIR) - (0.25*NIR + 2.75*SWIR) | Robust in arid / shadow-free regions |

Water pixels are positive for NDWI and MNDWI; positive for AWEInsh.


In [ ]:
def compute_indices(scene):
    """
    Compute NDWI, MNDWI, AWEInsh from an odc-stac scene.
    Returns three 2-D float32 numpy arrays.
    Pixels where all bands are zero (fill/nodata) are masked as NaN.
    """
    g  = scene.green.values.astype(np.float32)
    n  = scene.nir08.values.astype(np.float32)
    sw = scene.swir16.values.astype(np.float32)
    eps = 1e-9

    ndwi  = (g - n)  / (g + n  + eps)
    mndwi = (g - sw) / (g + sw + eps)
    awei  = 4 * (g - sw) - (0.25 * n + 2.75 * sw)  # AWEInsh

    nodata = (g == 0) & (n == 0) & (sw == 0)
    ndwi[nodata]  = np.nan
    mndwi[nodata] = np.nan
    awei[nodata]  = np.nan

    return ndwi, mndwi, awei


all_scenes  = {**landsat_data, **s2_data}   # merged timeline
all_indices = {yr: compute_indices(sc) for yr, sc in sorted(all_scenes.items())}

print(f"Indices computed for years: {sorted(all_indices.keys())}")


### Check indices for a sample year

Water = bright blue (positive); desert/bare soil = red (negative).


In [ ]:
sample_year = min(all_indices.keys())
ndwi, mndwi, awei = all_indices[sample_year]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f"Spectral indices -- {sample_year}", fontsize=13)

for ax, arr, title, lo, hi in [
    (axes[0], ndwi,  "NDWI",   -0.5, 0.5),
    (axes[1], mndwi, "MNDWI",  -0.5, 0.5),
    (axes[2], awei,  "AWEInsh", -0.2, 0.2),
]:
    im = ax.imshow(arr, cmap="RdYlBu", vmin=lo, vmax=hi)
    ax.set_title(title, fontsize=11)
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()


## 6. Binary water / non-water classification

A majority-vote rule is applied: a pixel is classified as **water (1)** when
at least 2 of the 3 indices exceed their threshold.  
This ensemble approach is more robust than any single index.

| Index | Water threshold |
|---|---|
| NDWI | > 0 |
| MNDWI | > 0 |
| AWEInsh | > 0 |


In [ ]:
NDWI_T  = 0.0
MNDWI_T = 0.0
AWEI_T  = 0.0


def classify_water(ndwi, mndwi, awei):
    """
    Majority-vote binary water mask.
    Returns uint8 array: 1 = water, 0 = non-water, 255 = nodata.
    """
    votes = (
        (ndwi  > NDWI_T ).astype(np.uint8) +
        (mndwi > MNDWI_T).astype(np.uint8) +
        (awei  > AWEI_T ).astype(np.uint8)
    )
    water = (votes >= 2).astype(np.uint8)
    water[np.isnan(ndwi) | np.isnan(mndwi) | np.isnan(awei)] = 255
    return water


pixel_area_km2 = (RESOLUTION / 1000) ** 2
water_masks    = {}
water_area_km2 = {}

for year, (ndwi, mndwi, awei) in sorted(all_indices.items()):
    mask = classify_water(ndwi, mndwi, awei)
    water_masks[year]    = mask
    water_area_km2[year] = int(np.sum(mask == 1)) * pixel_area_km2
    sensor = "LS" if year < 2016 else "S2"
    print(f"{year} [{sensor}]  water: {water_area_km2[year]:7,.0f} km2")


## 7. Water extent maps -- all years


In [ ]:
years = sorted(water_masks.keys())
ncols = 5
nrows = (len(years) + ncols - 1) // ncols

cmap_water = mcolors.ListedColormap(["#c9a96e", "#1565c0"])  # sand / water

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 3.5))
axes = axes.flatten()

for ax, year in zip(axes, years):
    disp = water_masks[year].copy().astype(float)
    disp[disp == 255] = np.nan
    ax.imshow(disp, cmap=cmap_water, vmin=0, vmax=1, interpolation="none")
    sensor = "LS" if year < 2016 else "S2"
    ax.set_title(f"{year} [{sensor}]\n{water_area_km2[year]:,.0f} km2", fontsize=9)
    ax.axis("off")

for ax in axes[len(years):]:
    ax.set_visible(False)

fig.legend(
    handles=[
        mpatches.Patch(color="#1565c0", label="Water"),
        mpatches.Patch(color="#c9a96e", label="Non-water"),
    ],
    loc="lower right", fontsize=10, framealpha=0.9,
)
fig.suptitle("Aral Sea -- water extent 2005-2024", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


## 8. Water area time series


In [ ]:
ls_yrs = sorted(y for y in water_area_km2 if y < 2016)
s2_yrs = sorted(y for y in water_area_km2 if y >= 2016)

fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(ls_yrs, [water_area_km2[y] for y in ls_yrs],
        "o-", color="#e07b39", lw=2, ms=7, label="Landsat C2 L2")
ax.plot(s2_yrs, [water_area_km2[y] for y in s2_yrs],
        "s-", color="#1565c0", lw=2, ms=7, label="Sentinel-2 L2A")
ax.axvspan(2015.5, 2016.5, color="grey", alpha=0.15, label="Sensor transition")

ax.set_xlabel("Year", fontsize=12)
ax.set_ylabel("Water area (km2)", fontsize=12)
ax.set_title("Aral Sea -- open water area estimate 2005-2024", fontsize=13)
ax.legend(fontsize=11)
ax.grid(axis="y", ls="--", alpha=0.5)
ax.set_xticks(years)
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

first, last = years[0], years[-1]
delta = water_area_km2[last] - water_area_km2[first]
print(f"{first}: {water_area_km2[first]:,.0f} km2")
print(f"{last}:  {water_area_km2[last]:,.0f} km2")
print(f"Change: {delta:+,.0f} km2  ({delta / water_area_km2[first] * 100:+.1f}%)")


## 9. Export timelapse GIF

Each frame = one year. The GIF loops continuously.
Output: `aral_sea_timelapse.gif`


In [ ]:
frames = []
cmap_gif = mcolors.ListedColormap(["#c9a96e", "#1565c0"])

for year in sorted(water_masks.keys()):
    disp = water_masks[year].copy().astype(float)
    disp[disp == 255] = np.nan

    fig, ax = plt.subplots(figsize=(7, 5), dpi=120)
    ax.imshow(disp, cmap=cmap_gif, vmin=0, vmax=1, interpolation="bilinear")
    sensor = "Landsat" if year < 2016 else "Sentinel-2"
    ax.set_title(
        f"Aral Sea  |  {year}  [{sensor}]\nWater area: {water_area_km2[year]:,.0f} km2",
        fontsize=12, pad=8,
    )
    ax.axis("off")
    ax.legend(
        handles=[
            mpatches.Patch(color="#1565c0", label="Water"),
            mpatches.Patch(color="#c9a96e", label="Non-water"),
        ],
        loc="lower right", fontsize=9, framealpha=0.9,
    )

    buf = BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    frames.append(iio.imread(buf))

out = "aral_sea_timelapse.gif"
iio.imwrite(out, frames, loop=0, duration=700)  # 700 ms per frame
print(f"Saved {len(frames)}-frame GIF -> {os.path.abspath(out)}")
